# Neural Networks Workshop
## Recognising Sea Creature Doodles 🦈🐬🐙🦀

We'll be using Google's **Quick, Draw!** dataset: millions of doodles people drew in 20 seconds while a game tried to guess what they were. We'll look t four sea animals: **shark, dolphin, octopus and crab**.

Each doodle is a **28 × 28 grayscale image** (**784 numbers**, one per pixel, each from 0 to 255). Our goal is to **predict which of the 4 animals a doodle is**. There are 4 possible answers, so this is **multi-class classification**.

---

### Plan

1. **Load and look:** shapes, class balance, what the doodles look like
2. **Prep:** scale the pixels, split into train and test
3. **Baseline:** logistic regression (a neural network with **no hidden layers**)
4. **Neural network:** add hidden layers (784 → 128 → 64 → 4, like the slides)
5. **How many layers?** loop over network sizes and watch train vs. validation accuracy
6. **Evaluate:** test accuracy, confusion matrix, the doodles it gets wrong
7. **Draw your own!** a live drawing app that runs the model on your doodles

### How this notebook works

- ✏️ **Write:** type the code yourself in the empty cell below the instructions
- 💬 **Discuss:** talk it over with the people next to you, and jot down an answer
- 👀 **Observe:** read and look at the output, no code to write
- Cells that are already filled in: just run them (Shift + Enter)

In [3]:
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt

import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)

CLASSES = ["shark", "dolphin", "octopus", "crab"]
N_PER_CLASS = 10_000
BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

### Get the data

Run this cell.

It downloads one file per animal, then keeps a random **10,000 doodles of each**. Every file has over 100,000 drawings, but we'll look at 40,000 total (enough for us to get good results and keep training fast).

- `X`: one row per doodle, 784 columns (the pixels)
- `y`: the label, a number 0 to 3 that indexes into `CLASSES` (0 = shark, 1 = dolphin, 2 = octopus, 3 = crab)

## 1. Load and look

Each row of `X` is **one doodle**, flattened from a 28 × 28 grid into one long row of 784 pixels.

In [ ]:
rng = np.random.default_rng(42)
X_parts, y_parts = [], []

for label, name in enumerate(CLASSES):
    path = f"{name}.npy"
    if not os.path.exists(path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(BASE_URL + name + ".npy", path)
    drawings = np.load(path, mmap_mode="r")
    keep = rng.choice(len(drawings), size=N_PER_CLASS, replace=False)
    X_parts.append(np.asarray(drawings[np.sort(keep)]))
    y_parts.append(np.full(N_PER_CLASS, label))
    print(f"{name:8s}: {len(drawings):,} drawings available, kept {N_PER_CLASS:,}")

X = np.concatenate(X_parts)
y = np.concatenate(y_parts)

✏️ **Check the shapes**

Write these three lines:
1. `print(X.shape, y.shape)` to see how many rows and columns we have
2. `print(X.dtype, X.min(), X.max())` to see what kind of numbers the pixels are
3. `print(np.bincount(y))` to count how many doodles there are of each label (0, 1, 2, 3)

In [ ]:
# YOUR CODE

👀 **Observe**
- `uint8` means whole numbers from 0 to 255.
- Exactly 10,000 of each animal, so the classes are **balanced**. Guessing at random would be right **25%** of the time. That's our "dumb" baseline.

✏️ **Look at one doodle**

1. `plt.imshow(X[0].reshape(28, 28), cmap="gray_r")` (`gray_r` draws ink as black on white)
2. `plt.title(CLASSES[y[0]])` to show its label
3. `plt.show()`

Try a few other row numbers instead of `0`.

In [ ]:
plt.imshow(X[0].reshape(28, 28), cmap="gray_r")
plt.title(CLASSES[y[0]])
plt.show()

Here are 8 random doodles of each animal:

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 8, figsize=(10, 5.5))
for row, name in enumerate(CLASSES):
    idx = rng.choice(np.where(y == row)[0], size=8, replace=False)
    for col, i in enumerate(idx):
        ax = axes[row, col]
        ax.imshow(X[i].reshape(28, 28), cmap="gray_r")
        ax.set_xticks([]); ax.set_yticks([])
    axes[row, 0].set_ylabel(name, fontsize=12)
plt.tight_layout()
plt.show()

💬 **Discuss**
- Which two animals do you think the model will **mix up** the most? Why?
- Some doodles are barely recognisable (people only had 20 seconds! plus, drawing an animal from memory is often harder than it sounds). What accuracy do you think is realistic? Would *you* get 100%?

*Your answer:*

## 2. Prep

Neural networks train best when inputs are **small numbers on the same scale**. Pixels go from 0 to 255, so we divide by 255 to squash them into 0 to 1. (Same idea as scaling features before regression.)

✏️ **Scale the pixels**

1. `X_scaled = X.astype("float32") / 255.0`
2. `print(X_scaled.min(), X_scaled.max())` to check it worked

In [ ]:
# YOUR CODE

✏️ **Split into train and test**

Hold back 20% as a **test set** that we only look at at the very end. `stratify=y` keeps all 4 animals at 25% in both halves.

1. `X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)`
2. `print(X_train.shape, X_test.shape)`

In [ ]:
# YOUR CODE

👀 **Observe: we won't cheat**

We'll compare models using a **validation set** rather than our test set: Keras will hold back 20% of the *training* data (`validation_split=0.2`) and report accuracy on it after every epoch.

So we have three groups:

| Set | Size | Used for |
|---|---|---|
| train | 25,600 | the model learns the weights from these |
| validation | 6,400 | we compare models and settings with these |
| test | 8,000 | locked away until section 6 |

## 3. Baseline: logistic regression

Logistic regression is just a neural network with **no hidden layers**: each of the 784 pixels connects straight to the 4 outputs. **Softmax** turns the 4 outputs into probabilities that add up to 1.

✏️ **Build the model**

`keras.Sequential` stacks layers in order. Type:
```python
logreg = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(4, activation="softmax"),
])
logreg.summary()
```
- `keras.Input(shape=(784,))`: each input is a row of 784 numbers
- `layers.Dense(4, ...)`: 4 output neurons, one per animal, each connected to **every** pixel ("dense")

In [ ]:
# YOUR CODE

💬 **Check the parameter count.** The summary says **3,140** parameters. Where does that number come from? (Hint: each of the 4 outputs has one weight per pixel, plus one bias.)

*Your answer:*

👀 **Observe: same idea, different syntax**

You've already seen logistic regression in scikit-learn. There it's:

```python
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
```

The Keras version below learns **the same kind of model**: one weight per pixel per animal, plus a bias, with softmax turning the 4 outputs into probabilities. The difference is that Keras makes you **build the layers yourself**, so you can see what's inside.

Why switch to Keras? scikit-learn is great for classic models (regression, trees, and so on), but it isn't built for **deeper neural networks**. It has a basic `MLPClassifier`, but it can't use a GPU and only supports simple layer types. Libraries like **Keras** (and PyTorch) are what people actually use for deep learning. Once logistic regression is written as layers, turning it into a neural network takes just **two more lines** (section 4).


✏️ **Compile: choose how it learns**

```python
logreg.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
```
- `optimizer="adam"`: a popular version of gradient descent that adjusts its own step size
- `loss="sparse_categorical_crossentropy"`: the loss for multi-class problems ("sparse" means our labels are plain numbers 0 to 3)
- `metrics=["accuracy"]`: also print accuracy, so we can read it

In [ ]:
# YOUR CODE

✏️ **Train it**

```python
history_logreg = logreg.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
)
```
- `epochs=10`: go through the training data 10 times
- `batch_size=128`: update the weights after every 128 doodles
- `validation_split=0.2`: hold back 20% of the training data to check on after each epoch

Look at `accuracy` (training) and `val_accuracy` (validation) as it runs.

In [ ]:
# YOUR CODE

### What did logistic regression learn?

Each output has one weight per pixel, so we can fold the 784 weights back into a 28 × 28 picture. **Red** pixels push towards that animal when there's ink there; **blue** pixels push away from it.

In [ ]:
W = logreg.layers[0].get_weights()[0]
lim = np.abs(W).max()

fig, axes = plt.subplots(1, len(CLASSES), figsize=(12, 3.2))
for k, ax in enumerate(axes):
    im = ax.imshow(W[:, k].reshape(28, 28), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(CLASSES[k])
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label="weight")
plt.show()

print(f"Logistic regression validation accuracy: {history_logreg.history['val_accuracy'][-1]:.1%}")

💬 **Discuss**
- Can you see a rough "template" of each animal in its weight map? Which one is clearest?
- Logistic regression can only add up "ink here is good, ink there is bad". Why is that a problem when people draw the **same animal in different positions or sizes**?
- How does its accuracy compare to the 25% random-guess baseline?

*Your answer:*

## 4. A neural network with hidden layers

Now the network from the slides: **784 → 128 → 64 → 4**. The hidden layers use **ReLU** (keep positive values, turn negatives into 0). That's what lets the network learn combinations like "a curve here **and** a fin there", instead of one template per animal.

The code is the same as before with two extra lines.

✏️ **Build it**

```python
model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(4, activation="softmax"),
])
model.summary()
```

In [ ]:
# YOUR CODE

👀 **Observe:** **108,996** parameters, about 35× more than logistic regression, and the same number as on the slides. Almost all of them are in the first layer: 784 × 128 + 128 = 100,480.

✏️ **Compile and train**

Same as before, just with `model` instead of `logreg`, and 15 epochs:
```python
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.2,
)
```

In [ ]:
# YOUR CODE

Let's plot how accuracy changed over the epochs, next to logistic regression:

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(history.history["accuracy"], label="network: train", color="tab:blue")
plt.plot(history.history["val_accuracy"], label="network: validation", color="tab:blue", linestyle="--")
plt.plot(history_logreg.history["accuracy"], label="log. regression: train", color="tab:gray")
plt.plot(history_logreg.history["val_accuracy"], label="log. regression: validation", color="tab:gray", linestyle="--")
plt.axhline(0.25, color="red", linestyle=":", label="random guess (25%)")
plt.xlabel("Epoch (0 = after the first pass)")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

💬 **Discuss**
- How much better is the network than logistic regression on **validation**?
- Look at the blue lines. Does training accuracy keep climbing while validation flattens out (or drops)? That gap is **overfitting**, the same thing we saw with deep trees.
- If you had to pick a number of epochs, which would you pick from this plot? (This is exactly what the validation set is for.)

*Your answer:*

## 5. How many layers?

The "knob" is the **size of the network**. We'll train 5 networks, from no hidden layers (logistic regression again) up to three big hidden layers, and record train and validation accuracy for each.

Run this cell (it'll takes a minute or two). Each entry in `layer_options` is a list of hidden layer sizes.

In [ ]:
layer_options = [[], [16], [64], [128, 64], [256, 128, 64]]
labels = ["none", "16", "64", "128-64", "256-128-64"]
train_scores, val_scores = [], []

for hidden in layer_options:
    keras.utils.set_random_seed(42)
    net = keras.Sequential([keras.Input(shape=(784,))])
    for size in hidden:
        net.add(layers.Dense(size, activation="relu"))
    net.add(layers.Dense(4, activation="softmax"))
    net.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    h = net.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.2, verbose=0)
    train_scores.append(h.history["accuracy"][-1])
    val_scores.append(h.history["val_accuracy"][-1])
    print(f"hidden={str(hidden):16s} params={net.count_params():>8,}  "
          f"train={train_scores[-1]:.1%}  val={val_scores[-1]:.1%}")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(labels, train_scores, marker="o", label="train")
plt.plot(labels, val_scores, marker="o", label="validation")
plt.axhline(0.25, color="red", linestyle=":", label="random guess (25%)")
plt.xlabel("Hidden layers")
plt.ylabel("Accuracy after 10 epochs")
plt.title("Bigger network: better fit, but does it generalise?")
plt.legend()
plt.show()

👀 **Observe**
- As the network grows, **train** accuracy should keep going up, because more parameters make it easier to memorise. Watch whether **validation** keeps up, levels off, or falls back: a growing gap between the lines is overfitting.
- The exact numbers move a little from run to run (random starting weights), so don't read too much into differences of half a percent.

💬 **Discuss**
- Which network would you pick, and why? Think about accuracy **and** size (the biggest one has over 200,000 parameters).
- Besides adding layers, what other knobs could you turn? (The challenges at the end have a few ideas.)

*Your answer:*

## 6. Evaluate on the test set

We've made our choices using validation, so now we open the test set, **once**, to get an honest score for our 128-64 `model`.

`model.predict` returns **4 probabilities per doodle** (one per animal). The prediction is whichever is biggest; `argmax(axis=1)` gives the position of the biggest value in each row.

✏️ **Predict and score**

1. `probs = model.predict(X_test)`
2. `print(probs[0].round(3))` to see the 4 probabilities for the first test doodle
3. `y_pred = probs.argmax(axis=1)`
4. `print(f"Test accuracy: {accuracy_score(y_test, y_pred):.1%}")`

In [ ]:
# YOUR CODE

In [ ]:
logreg_pred = logreg.predict(X_test, verbose=0).argmax(axis=1)
print(f"Random guessing:     25.0%")
print(f"Logistic regression: {accuracy_score(y_test, logreg_pred):.1%}")
print(f"Neural network:      {accuracy_score(y_test, y_pred):.1%}")

### Which animals get mixed up?

Rows are the **true** animal, columns are what the model **predicted**. The diagonal is correct answers.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=CLASSES,
    cmap="Blues",
    values_format=",",
)
plt.title("Neural network on the test set")
plt.show()

And here are some doodles it got **wrong**, with what it guessed and how confident it was:

In [ ]:
wrong = np.where(y_pred != y_test)[0]
show = rng.choice(wrong, size=min(15, len(wrong)), replace=False)

fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for ax, i in zip(axes.flat, show):
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{CLASSES[y_test[i]]} → {CLASSES[y_pred[i]]}\n({probs[i].max():.0%} sure)", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("true → predicted", y=1.0)
plt.tight_layout()
plt.show()

💬 **Discuss**
- Which pair gets confused the most? Was it the one you guessed in section 1?
- Look at the wrong ones. Which are the model's fault, and/or which would **you** have gotten wrong?

*Your answer:*

## 7. Draw your own! 🎨

Time to test the model on **your** doodles. The app below takes your drawing, converts it into the same format as the Quick, Draw! data (ink on blank, cropped, centred, shrunk to 28 × 28) and feeds those 784 numbers to `model`.

**Run the three cells below.** If you fell behind, the first one trains a quick model so you can still play.

In [ ]:
if "model" not in globals() or not hasattr(model, "predict"):
    if "X_train" not in globals():
        X_scaled = X.astype("float32") / 255.0
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, stratify=y, random_state=42)
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(4, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X_train, y_train, epochs=5, batch_size=128, verbose=0)
    print("Trained a catch-up model.")
else:
    print("Using your trained model.")

In [ ]:
from PIL import Image, ImageFilter
import gradio as gr

def preprocess(drawing):
    if drawing is None:
        return None
    img = drawing["composite"] if isinstance(drawing, dict) else drawing
    if img is None:
        return None
    img = np.asarray(img).astype("float32")
    if img.ndim == 2:
        ink = 255.0 - img
    else:
        rgb, alpha = img[..., :3], (img[..., 3] / 255.0 if img.shape[-1] == 4 else 1.0)
        ink = (255.0 - rgb.mean(axis=-1)) * alpha

    rows, cols = np.where(ink > 30)
    if len(rows) == 0:
        return None
    ink = ink[rows.min():rows.max() + 1, cols.min():cols.max() + 1]

    h, w = ink.shape
    side = int(max(h, w) * 1.1) + 1
    square = np.zeros((side, side), dtype="float32")
    top, left = (side - h) // 2, (side - w) // 2
    square[top:top + h, left:left + w] = ink

    k = max(3, int(side / 28) | 1)
    big = Image.fromarray(square).filter(ImageFilter.MaxFilter(k))
    small = big.resize((28, 28), Image.Resampling.BOX)
    small = np.asarray(small, dtype="float32")
    small = small / small.max()
    return small.reshape(784)


def classify(drawing):
    x = preprocess(drawing)
    if x is None:
        return {}, None
    p = model.predict(x[None, :], verbose=0)[0]
    seen = Image.fromarray(((1 - x.reshape(28, 28)) * 255).astype("uint8")).resize((224, 224), Image.Resampling.NEAREST)
    return {name: float(prob) for name, prob in zip(CLASSES, p)}, seen

In [ ]:
demo = gr.Interface(
    fn=classify,
    inputs=gr.Sketchpad(
        type="numpy",
        image_mode="RGBA",
        canvas_size=(400, 400),
        brush=gr.Brush(default_size=14, colors=["#000000"], color_mode="fixed"),
        label="Draw a shark, dolphin, octopus or crab",
    ),
    outputs=[
        gr.Label(num_top_classes=4, label="Model's guess"),
        gr.Image(label="What the model sees (28 × 28)"),
    ],
    live=True,
    title="Sea Creature Doodle Classifier",
)
demo.launch(share=True)

👀 **Try this:** `share=True` prints a public link (`...gradio.live`). Open it on your **phone** and draw with your finger!

If a drawing you think is obvious gets the wrong answer, look at **"What the model sees"**. Is your drawing much thinner, thicker or smaller than the Quick, Draw! doodles in section 1?

💬 **Discuss**
- Find a drawing that **fools** the model. What did you change?
- Now draw something that **isn't** a sea animal: a house, a sun, your name. What does the model say, and how sure is it?
- Why can't the model ever answer "none of these"? What would you change so it could?

*Your answer:*

## 8. Wrap-up

**Takeaways**
1. **An image is just numbers.** A 28 × 28 doodle is a row of 784 pixel values, and every pixel is a feature.
2. **Logistic regression is a network with no hidden layers.** It learns one "template" per class, which struggles when things move around or change size.
3. **Hidden layers + ReLU let the network combine features.** Same code, two extra lines, and a big jump in accuracy.
4. **Bigger isn't always better.** More layers fit the training data better, but watch the gap to validation, just like `max_depth`.
5. **Choose with validation, report with test.** The test set stays locked until the very end, so its score is honest.
6. **Softmax always picks something.** The model has only ever seen 4 animals, so a house still comes out as (say) "90% crab". Confidence is not the same as correctness.

### Challenges
- **Train longer.** Change `epochs=15` to `epochs=40` for `model` and re-plot the curves. When does validation stop improving? Look up `keras.callbacks.EarlyStopping`.
- **Add Dropout.** Put `layers.Dropout(0.3)` after each hidden `Dense` layer. Does the gap between train and validation shrink?
- **Add a 5th animal.** Add `"whale"` (or `"fish"`, `"lobster"`... any name from the [Quick, Draw! category list](https://github.com/googlecreativelab/quickdraw-dataset/blob/master/categories.txt)) to `CLASSES`, and replace the `4` in the output layer with `len(CLASSES)`. Which animal does it get confused with?
- **Use a CNN.** Our network sees pixels as one long row, so it doesn't know which pixels are **neighbours**. A convolutional network does. Try:
  ```python
  cnn = keras.Sequential([
      keras.Input(shape=(28, 28, 1)),
      layers.Conv2D(32, 3, activation="relu"),
      layers.MaxPooling2D(),
      layers.Conv2D(64, 3, activation="relu"),
      layers.MaxPooling2D(),
      layers.Flatten(),
      layers.Dense(64, activation="relu"),
      layers.Dense(4, activation="softmax"),
  ])
  ```
  and train it on `X_train.reshape(-1, 28, 28, 1)`. (To use it in the drawing app, reshape `x` the same way in `classify`.)